# GEE Dataset Cleaning Notebook

This notebook cleans `GEE-Dataset.csv` by applying the following steps:

1. **Remove NaN rows** — drop any row where all spectral value columns are NaN
2. **Drop IMAGE_COUNT column**
3. **Clean LAKE_NAME** — strip noise words: `kere`, `lake`, `tank`, `new`, and `(Un-named Lake on map)`
4. **Keep first meaningful word** — handles `B. Channasandra`, `Lake-1/2`, `Kengeri 2`, etc.
5. **Split multi-name rows** — e.g. `Nagareshvara Nagenahalli Kere` → two rows: `Nagareshvara` and `Nagenahalli`

In [15]:
import pandas as pd
import re

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('GEE-Dataset.csv')
print('Original shape:', df.shape)
df.head(10)

Original shape: (4525, 7)


,LAKE_NAME,DATE,NDCI_Chla,NDTI,NDWI,NDSSI,IMAGE_COUNT
0,Gunduru Lake,07/23,NaN,NaN,NaN,NaN,0
1,Doddebele,07/23,NaN,NaN,NaN,NaN,0
2,Somapura,07/23,NaN,NaN,NaN,NaN,0
3,Mailasandra,07/23,NaN,NaN,NaN,NaN,0
4,Kengeri 1,07/23,NaN,NaN,NaN,NaN,0
5,Kengeri 2,07/23,NaN,NaN,NaN,NaN,0
6,Chenvinayanahalli,07/23,NaN,NaN,NaN,NaN,0
7,Annamma kere,07/23,NaN,NaN,NaN,NaN,0
8,Deevatigeramnahalli kere,07/23,NaN,NaN,NaN,NaN,0
9,Devarakere,07/23,NaN,NaN,NaN,NaN,0


In [16]:
# ── Step 1: Remove rows where ALL spectral value columns are NaN ──────────────
value_cols = ['NDCI_Chla', 'NDTI', 'NDWI', 'NDSSI']
df = df.dropna(subset=value_cols, how='all').reset_index(drop=True)
print('After removing all-NaN rows:', df.shape)

After removing all-NaN rows: (3508, 7)


In [17]:
# ── Step 2: Drop IMAGE_COUNT column ──────────────────────────────────────────
if 'IMAGE_COUNT' in df.columns:
    df = df.drop(columns=['IMAGE_COUNT'])
print('Columns:', df.columns.tolist())

Columns: ['LAKE_NAME', 'DATE', 'NDCI_Chla', 'NDTI', 'NDWI', 'NDSSI']


In [18]:
# ── Helper: extract meaningful words from a lake name ─────────────────────────
#
# Processing pipeline for each name:
#  1. Remove "(Un-named Lake on map)" suffix
#  2. Detect and preserve "B." / "B. " prefix (e.g. B. Channasandra)
#  3. Keep only the first part when slash-separated (e.g. "X kere / Y kere")
#  4. Strip trailing numeric suffixes: -1, -2, 1, 2
#  5. Strip inline kere1/kere2 suffixes
#  6. Remove all noise words (kere, lake, tank, new, katte, govt, etc.)
#  7. Return list of remaining meaningful words
#     -> 1 word  : keep as-is
#     -> 2+ words: caller will split into separate rows

NOISE_WORDS = [
    r'\bkere\b', r'\blake\b', r'\btank\b', r'\bnew\b',
    r'\bkatte\b', r'\bkunte\b', r'\bagrahara\b', r'\bgudde\b',
    r'\bgovt\.?\b', r'\blayout\b', r'\bappartment\b',
    r'\bcorridor\b', r'\bgramadakere\b', r'\bhoodikere\b',
    r'\bpark\b', r'\bnagar\b',
]

def get_meaningful_words(name: str) -> list:
    """Return cleaned, meaningful name-tokens for a raw LAKE_NAME string."""
    if not isinstance(name, str):
        return [str(name)]

    s = name.strip()

    # 1. Remove (Un-named Lake on map)
    s = re.sub(r'\s*\(Un-named Lake on map\)', '', s, flags=re.IGNORECASE).strip()

    # 2. Detect "B. " or "B." prefix -- capture and strip temporarily
    b_prefix = ''
    bm = re.match(r'^([A-Z]\.\s*)', s)
    if bm:
        b_prefix = bm.group(1).rstrip()  # "B."
        s = s[bm.end():].strip()

    # 3. Slash-separated alternatives -- keep first part only
    s = s.split('/')[0].strip()

    # 4. Strip trailing numeric suffix: -2, -1, 2, 1 ...
    s = re.sub(r'[-\s]?\d+$', '', s).strip()

    # 5. Strip inline kere1/kere2 suffixes
    s = re.sub(r'(kere|lake)\d+', r'\1', s, flags=re.IGNORECASE)

    # 6. Remove all noise words
    for pattern in NOISE_WORDS:
        s = re.sub(pattern, '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s{2,}', ' ', s).strip()

    # 7. Split into words, reattach B. prefix to first word
    words = [w for w in s.split() if w]
    if b_prefix:
        if words:
            words[0] = b_prefix + ' ' + words[0]   # -> "B. Channasandra"
        else:
            words = [b_prefix]

    return words if words else [name.strip()]  # fallback: return original


# ── Quick sanity check on tricky cases ───────────────────────────────────────
test_cases = [
    ('Nagareshvara Nagenahalli Kere',      ['Nagareshvara', 'Nagenahalli']),
    ('B. Channasandra lake',               ['B. Channasandra']),
    ('B.Narayanapura kere',                ['B. Narayanapura']),
    ('Kengeri 2',                          ['Kengeri']),
    ('Mallasandra Lake-2',                 ['Mallasandra']),
    ('Nagarabhavi (Un-named Lake on map)', ['Nagarabhavi']),
    ('Sankey Tank',                        ['Sankey']),
    ('Shivapura New tank',                 ['Shivapura']),
    ('Halagevaderahalli kere-2',           ['Halagevaderahalli']),
    ('Vijinapura kere2',                   ['Vijinapura']),
    ('Gangashetti kere/ Devasandra kere',  ['Gangashetti']),
    ('Muppatu Kavalu Hosa Kere',           ['Muppatu', 'Kavalu', 'Hosa']),
]

all_pass = True
for raw, expected in test_cases:
    result = get_meaningful_words(raw)
    status = 'PASS' if result == expected else 'FAIL'
    if result != expected:
        all_pass = False
    print(f"[{status}]  {raw!r:50s}  ->  {result}")

print('\nAll tests passed!' if all_pass else '\nSome tests failed -- review logic above.')

[PASS]  'Nagareshvara Nagenahalli Kere'                     ->  ['Nagareshvara', 'Nagenahalli']
[PASS]  'B. Channasandra lake'                              ->  ['B. Channasandra']
[PASS]  'B.Narayanapura kere'                               ->  ['B. Narayanapura']
[PASS]  'Kengeri 2'                                         ->  ['Kengeri']
[PASS]  'Mallasandra Lake-2'                                ->  ['Mallasandra']
[PASS]  'Nagarabhavi (Un-named Lake on map)'                ->  ['Nagarabhavi']
[PASS]  'Sankey Tank'                                       ->  ['Sankey']
[PASS]  'Shivapura New tank'                                ->  ['Shivapura']
[PASS]  'Halagevaderahalli kere-2'                          ->  ['Halagevaderahalli']
[PASS]  'Vijinapura kere2'                                  ->  ['Vijinapura']
[PASS]  'Gangashetti kere/ Devasandra kere'                 ->  ['Gangashetti']
[PASS]  'Muppatu Kavalu Hosa Kere'                          ->  ['Muppatu', 'Kavalu', 'Hosa']

All tes

In [19]:
# ── Steps 3-6: Clean LAKE_NAME and expand multi-name rows ─────────────────────
#
# For each row:
#   * 1 meaningful word  -> update LAKE_NAME in-place
#   * 2+ meaningful words -> duplicate the row for each word (values stay same)

expanded_rows = []

for _, row in df.iterrows():
    words = get_meaningful_words(row['LAKE_NAME'])
    for w in words:
        new_row = row.copy()
        new_row['LAKE_NAME'] = w
        expanded_rows.append(new_row)

df_clean = pd.DataFrame(expanded_rows).reset_index(drop=True)

print('Cleaned shape :', df_clean.shape)
print('Unique lakes  :', df_clean['LAKE_NAME'].nunique())

# Verify the Nagareshvara / Nagenahalli split
for name in ['Nagareshvara', 'Nagenahalli']:
    count = (df_clean['LAKE_NAME'] == name).sum()
    print(f'  Rows for {name}: {count}')

df_clean.head(20)

Cleaned shape : (3997, 6)
Unique lakes  : 181
  Rows for Nagareshvara: 20
  Rows for Nagenahalli: 20


,LAKE_NAME,DATE,NDCI_Chla,NDTI,NDWI,NDSSI
0,Gunduru,08/23,0.199080,-0.072871,-0.287658,-0.433058
1,Doddebele,08/23,0.253024,-0.141050,-0.146894,-0.247766
2,Somapura,08/23,0.142168,-0.010070,-0.333981,-0.410639
3,Mailasandra,08/23,0.285452,-0.110162,-0.579786,-0.687140
4,Kengeri,08/23,0.118037,-0.018408,-0.336622,-0.406101
5,Kengeri,08/23,0.224355,-0.070850,-0.453158,-0.548267
6,Chenvinayanahalli,08/23,0.172535,-0.037215,-0.346784,-0.390230
7,Annamma,08/23,0.237318,-0.127675,-0.150886,-0.359802
8,Gubbalal,08/23,0.175994,-0.028662,-0.373403,-0.492863
9,Halagevaderahalli,08/23,0.168529,-0.038668,-0.432660,-0.475650


In [20]:
# ── All unique lake names after cleaning ─────────────────────────────────────
unique_names = sorted(df_clean['LAKE_NAME'].unique())
print(f'Total unique lake names: {len(unique_names)}\n')
for n in unique_names:
    print(' ', n)
df_clean = df_clean[df_clean['LAKE_NAME'].str.strip() != "."]

Total unique lake names: 181

  ,Mahadevepura
  .
  Abbigere
  Acchanakere
  Achanakere
  Agahara
  Agara
  Agrahara Lake
  Ajjegowdana
  Akashaya
  Alahalli
  Allasandra
  Amanikere
  Ambalipura
  Amruthahalli
  Amrutnagar
  Andhrahalli
  Annamma
  Arekere
  Atturu
  Avalahali
  B. Channasandra
  B. Narayanapura
  Bagalagunte
  Basavanagar
  Basavanapura
  Begur
  Bellandur
  Bellihalli
  Beratena
  Bhattarahalli
  Bhimmanakuppe
  Bhoganahalli
  Bommasandra
  Byappanahalli
  Byrasandra
  Carmelarm
  Chellaghatta
  Chellakere
  Chenvinayanahalli
  Chikere
  Chikkabanavara
  Chikkabegur
  Chikkabellanduru
  Chinnappanahalli
  Chokkanahalli
  Chunchanaghatta
  Dasarahalli
  Deevatigeramnahalli
  Devarabisanahalli
  Devarakere
  Dinnekere
  Dodda
  Doddabidarakallu
  Doddagubbi
  Doddakallasandra
  Doddakanneli
  Doddanekundi
  Doddebele
  Doraikere
  Elenahalli
  Gangashetti
  Gottigere
  Goudanakere
  Gramadakere
  Gubbalal
  Gunduru
  Gunjuruplaya
  Halagevaderahalli
  Handrahalli
  Ha

In [21]:
# ── Save cleaned dataset ──────────────────────────────────────────────────────
output_path = 'GEE-Dataset-Cleaned.csv'
df_clean.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')
df_clean

Saved to: GEE-Dataset-Cleaned.csv


,LAKE_NAME,DATE,NDCI_Chla,NDTI,NDWI,NDSSI
0,Gunduru,08/23,0.199080,-0.072871,-0.287658,-0.433058
1,Doddebele,08/23,0.253024,-0.141050,-0.146894,-0.247766
2,Somapura,08/23,0.142168,-0.010070,-0.333981,-0.410639
3,Mailasandra,08/23,0.285452,-0.110162,-0.579786,-0.687140
4,Kengeri,08/23,0.118037,-0.018408,-0.336622,-0.406101
...,...,...,...,...,...,...
3992,Acchanakere,11/25,0.323478,-0.345071,0.218217,-0.124288
3993,Varthur,11/25,0.224400,-0.172616,-0.025074,-0.203370
3994,Narasipura,11/25,0.162361,-0.188206,0.100589,-0.140285
3995,Narasipura,11/25,0.249301,-0.135239,-0.304986,-0.504832
